In [ ]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

pd.set_option("display.max_columns", None)
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
os.makedirs("models", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)


In [ ]:
def find_file(filename):
    candidates = [
        Path(filename),
        Path("data/processed") / filename,
        Path("../data/processed") / filename,
        Path("../../data/processed") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find '{filename}'. Looked in: {[str(c) for c in candidates]}\n"
        f"Fix: copy '{filename}' into the same folder as this notebook, "
        f"or into 'data/processed/' relative to the project root."
    )

train_df = pd.read_csv(find_file("train_features.csv"))
test_df = pd.read_csv(find_file("test_features.csv"))

print("train_df:", train_df.shape, " test_df:", test_df.shape)


In [ ]:
categorical_cols = ["main_category", "currency", "country", "launch_weekday"]
numeric_cols = [
    "usd_goal_real", "goal_log", "campaign_duration",
    "launch_month", "launch_quarter",
    "category_frequency", "country_success_rate",
]
feature_cols = categorical_cols + numeric_cols

X_train = train_df[feature_cols]
y_train = train_df["target"]
X_test = test_df[feature_cols]
y_test = test_df["target"]

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numeric_cols),
])

# Day 5 — Boosting Models


In [ ]:
boosting_models = {
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, max_depth=-1, learning_rate=0.05,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05,
        random_state=RANDOM_STATE, verbose=0,
    ),
}

fitted_boosting = {}
timing_records = []

for name, model in boosting_models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])

    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = pipe.predict(X_test)
    inference_time = time.perf_counter() - t0

    y_proba = pipe.predict_proba(X_test)[:, 1]

    fitted_boosting[name] = pipe
    timing_records.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "Train Time (s)": round(train_time, 3),
        "Inference Time (s)": round(inference_time, 4),
    })
    print(f"Trained: {name}  (train={train_time:.2f}s, infer={inference_time:.3f}s)")

boosting_results_df = pd.DataFrame(timing_records).sort_values("ROC-AUC", ascending=False).round(4)
boosting_results_df.to_csv("reports/boosting_model_comparison.csv", index=False)
boosting_results_df

### Compare against Day 4 baselines


In [ ]:
baseline_path = None
for candidate in [Path("reports/baseline_model_comparison.csv"),
                  Path("../reports/baseline_model_comparison.csv")]:
    if candidate.exists():
        baseline_path = candidate
        break

if baseline_path:
    baseline_results_df = pd.read_csv(baseline_path)
    combined_df = pd.concat([baseline_results_df, boosting_results_df], ignore_index=True)
    combined_df = combined_df.sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
    combined_df.to_csv("reports/full_model_comparison.csv", index=False)
    display(combined_df)
else:
    print("baseline_model_comparison.csv not found nearby -- skipping merge, "
          "boosting-only comparison is above.")


### Accuracy vs training time — visualize the tradeoff

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(boosting_results_df["Train Time (s)"], boosting_results_df["ROC-AUC"], s=120)

for _, row in boosting_results_df.iterrows():
    ax.annotate(row["Model"], (row["Train Time (s)"], row["ROC-AUC"]),
                textcoords="offset points", xytext=(8, 5))

ax.set_xlabel("Training Time (s)")
ax.set_ylabel("ROC-AUC")
ax.set_title("Boosting Models: Accuracy vs Training Time")
plt.tight_layout()
plt.savefig("reports/figures/boosting_accuracy_vs_time.png", dpi=300, bbox_inches="tight")
plt.show()


### Feature importance — all three boosters side by side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (name, pipe) in zip(axes, fitted_boosting.items()):
    feature_names = pipe.named_steps["preprocessor"].get_feature_names_out()
    importances = pipe.named_steps["model"].feature_importances_

    imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False).head(10)

    sns.barplot(data=imp_df, x="importance", y="feature", ax=ax)
    ax.set_title(name)

plt.tight_layout()
plt.savefig("reports/figures/boosting_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

# Day 6 — Hyperparameter Tuning

In [ ]:
best_boosting_name = boosting_results_df.iloc[0]["Model"]
print(f"Tuning: {best_boosting_name}")

base_model_lookup = {
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=RANDOM_STATE, verbose=0),
}
base_model = base_model_lookup[best_boosting_name]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### RandomizedSearchCV (broad search)

In [ ]:
param_prefix = "model__"

random_param_dist = {
    f"{param_prefix}n_estimators": [100, 200, 300, 500, 800],
    f"{param_prefix}max_depth": [3, 4, 5, 6, 8, 10],
    f"{param_prefix}learning_rate": [0.01, 0.02, 0.05, 0.08, 0.1, 0.15],
}
# CatBoost uses 'iterations'/'depth' instead of 'n_estimators'/'max_depth'
if best_boosting_name == "CatBoost":
    random_param_dist = {
        f"{param_prefix}iterations": [100, 200, 300, 500, 800],
        f"{param_prefix}depth": [3, 4, 5, 6, 8, 10],
        f"{param_prefix}learning_rate": [0.01, 0.02, 0.05, 0.08, 0.1, 0.15],
    }

tuning_pipe = Pipeline([("preprocessor", preprocessor), ("model", base_model)])

random_search = RandomizedSearchCV(
    tuning_pipe,
    param_distributions=random_param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

t0 = time.perf_counter()
random_search.fit(X_train, y_train)
random_search_time = time.perf_counter() - t0

print(f"RandomizedSearchCV done in {random_search_time:.1f}s")
print("Best params:", random_search.best_params_)
print("Best CV ROC-AUC:", round(random_search.best_score_, 4))

### GridSearchCV (narrow search around the best region)

In [ ]:
def neighbors(value, deltas):
    return sorted(set(v for v in [value + d for d in deltas] if v > 0))

best_params = random_search.best_params_
depth_key = f"{param_prefix}depth" if best_boosting_name == "CatBoost" else f"{param_prefix}max_depth"
est_key = f"{param_prefix}iterations" if best_boosting_name == "CatBoost" else f"{param_prefix}n_estimators"
lr_key = f"{param_prefix}learning_rate"

grid_param_dist = {
    est_key: neighbors(best_params[est_key], [-100, 0, 100]),
    depth_key: neighbors(best_params[depth_key], [-1, 0, 1]),
    lr_key: sorted(set([round(best_params[lr_key] * f, 3) for f in [0.7, 1.0, 1.3]])),
}

grid_search = GridSearchCV(
    tuning_pipe,
    param_grid=grid_param_dist,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

t0 = time.perf_counter()
grid_search.fit(X_train, y_train)
grid_search_time = time.perf_counter() - t0

print(f"GridSearchCV done in {grid_search_time:.1f}s")
print("Best params:", grid_search.best_params_)
print("Best CV ROC-AUC:", round(grid_search.best_score_, 4))


### Optuna (optional)

In [ ]:
try:
    import optuna
    from sklearn.model_selection import cross_val_score

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        if best_boosting_name == "CatBoost":
            params = {
                "iterations": trial.suggest_int("iterations", 100, 800),
                "depth": trial.suggest_int("depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            }
        else:
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 800),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            }
        model = base_model_lookup[best_boosting_name].__class__(**{**params,
            **({"eval_metric": "logloss", "random_state": RANDOM_STATE, "n_jobs": -1}
               if best_boosting_name == "XGBoost" else
               {"random_state": RANDOM_STATE, "n_jobs": -1, "verbose": -1}
               if best_boosting_name == "LightGBM" else
               {"random_state": RANDOM_STATE, "verbose": 0})
        })
        pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
        scores = cross_val_score(pipe, X_train, y_train, scoring="roc_auc", cv=cv, n_jobs=-1)
        return scores.mean()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20, show_progress_bar=False)

    print("Optuna best ROC-AUC:", round(study.best_value, 4))
    print("Optuna best params:", study.best_params)

except ImportError:
    print("optuna not installed -- skipping this cell. "
          "Run '%pip install optuna' in the setup cell above to enable it.")

## Final evaluation of the tuned model on the held-out test set

In [ ]:
tuned_pipeline = grid_search.best_estimator_

y_pred = tuned_pipeline.predict(X_test)
y_proba = tuned_pipeline.predict_proba(X_test)[:, 1]

tuned_results = {
    "Model": f"{best_boosting_name} (tuned)",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba),
}

print(f"Untuned {best_boosting_name} ROC-AUC: "
      f"{boosting_results_df.set_index('Model').loc[best_boosting_name, 'ROC-AUC']:.4f}")
print(f"Tuned {best_boosting_name} ROC-AUC:   {tuned_results['ROC-AUC']:.4f}")

pd.DataFrame([tuned_results]).round(4)

### Save the tuned model

In [ ]:
joblib.dump(tuned_pipeline, "models/boosting_tuned_best_model.pkl")
print("Saved to models/boosting_tuned_best_model.pkl")

import shutil
shutil.make_archive("boosting_outputs", "zip", "reports")

try:
    from google.colab import files
    files.download("boosting_outputs.zip")
    files.download("models/boosting_tuned_best_model.pkl")
except ImportError:
    print("Not running in Colab -- files saved locally:")
    print(" - boosting_outputs.zip")
    print(" - models/boosting_tuned_best_model.pkl")